# 05 - Model Training & Evaluation

Train a baseline classifier to predict whether a property is
near the beach, then evaluate on the held-out test split.

In [ ]:
import os
from pathlib import Path

# Mirror the same path convention used in the other notebooks
path = os.getcwd()
path = os.path.abspath(os.path.join(path, "..", "data"))

DATA_DIR      = Path(path)
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR    = DATA_DIR.parent / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE    = PROCESSED_DIR / "train.parquet"
TEST_FILE     = PROCESSED_DIR / "test.parquet"

TARGET_COLUMN = "label"
RANDOM_STATE  = 42

In [ ]:
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

train = pd.read_parquet(TRAIN_FILE)
test = pd.read_parquet(TEST_FILE)
target = TARGET_COLUMN

## Prepare feature matrices

In [ ]:
feature_cols = [c for c in train.columns if c != target]
# Use only numeric features for this baseline; encode categoricals as needed.
feature_cols = train[feature_cols].select_dtypes(include="number").columns.tolist()

X_train, y_train = train[feature_cols], train[target]
X_test, y_test = test[feature_cols], test[target]
print(f"Using {len(feature_cols)} features")

## Train baseline model

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)

## Evaluate

In [ ]:
preds = model.predict(X_test)
print(classification_report(y_test, preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, preds))

## Feature importance

In [ ]:
importances = (
    pd.Series(model.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
)
importances

## Persist model

In [ ]:
model_path = MODELS_DIR / "beach_predictor.joblib"
joblib.dump(model, model_path)
print(f"Saved model -> {model_path}")